In [12]:
# !python.exe -m pip install --upgrade pip
# !pip install sqlalchemy
# !pip install snowflake-sqlalchemy

In [2]:
import pandas as pd
import numpy as np
import json
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine

import os
from dotenv import load_dotenv

load_dotenv()

snowflake_connection_string = os.getenv('connection_string')

In [6]:

parts = snowflake_connection_string.split("//")[1].split("/")  
account = ".".join(parts[0].split('.')[:2]) 
user = parts[1].split('user=')[1].split('&')[0] 
password = parts[1].split('password=')[1].split('&')[0] 
database = parts[1].split('db=')[1].split('&')[0] 
warehouse = parts[1].split('warehouse=')[1].split('&')[0]  
role = parts[1].split('role=')[1]

engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    database = database,
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur = engine.connect()

def read_data_from_snowflake_table(cur,query):
    df = pd.read_sql(query, cur)
    return df


In [9]:
SPT = read_data_from_snowflake_table(cur,"select distinct product_cd from SPT")

In [41]:
grade_names = SPT['product_cd'].unique().tolist()
grade_names=set([grade.lower() for grade in grade_names])

## Please update the filename before run all

In [33]:
filename ="../../dependencies/unique_values_25_07_23.json"
f=open(filename,mode='r')
unique_values = json.load(f)
existing_grade_names = set(unique_values['GRADE'])

In [53]:
len(grade_names), len(existing_grade_names)

(1926, 1243)

In [54]:
updated_grade_names = list(existing_grade_names.union(grade_names))

In [55]:
len(updated_grade_names)

1981

In [59]:
unique_values['GRADE'] = updated_grade_names
with open("../../dependencies/unique_values.json", 'w') as file:
    json.dump(unique_values, file, indent=2)